In [ ]:
# 12.10 캐글 1위 전략 도입 버전

In [ ]:
# ============================================================================
# MercariLeaderAnalyzer - Kaggle 1위 전략 구현
# ============================================================================
# 전략: Ridge(텍스트) + LightGBM(카테고리/수치) 2-Stage 앙상블
# ============================================================================

import os
import sys
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from lightgbm import LGBMRegressor
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

# 프로젝트 루트
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


class MercariLeaderAnalyzer:
    """
    Mercari Price Prediction - Kaggle 1위 전략 구현
    
    핵심 전략:
    ---------
    1. **Stage 1-A: Ridge Regressor**
       - Input: TF-IDF 벡터화된 텍스트 (name + description)
       - Output: 텍스트 기반 가격 예측
    
    2. **Stage 1-B: LightGBM**
       - Input: 카테고리 인코딩 + 수치 피처
       - Output: 메타데이터 기반 가격 예측
    
    3. **Stage 2: Weighted Ensemble**
       - Ridge + LightGBM 예측값을 가중 평균
       - 최적 가중치는 검증 세트에서 탐색
    
    주요 차이점 (vs MercariSklearnAnalyzer):
    ----------------------------------------
    - TF-IDF와 카테고리를 분리하여 각각 전문 모델로 학습
    - Ridge는 텍스트에 강하고, LightGBM은 범주형에 강함
    - 간단한 앙상블로 각 모델의 장점 극대화
    """

    # __init__ start ###########################
    def __init__(
        self,
        random_state: int = 23,
        models_dir: str = "../models",
        results_dir: str = "../results",
        images_dir: str = "../images",
    ):
        """
        초기화
        
        Parameters:
        -----------
        random_state : int
            랜덤 시드
        models_dir, results_dir, images_dir : str
            저장 경로
        """
        self.random_state = random_state
        self.models_dir = models_dir
        self.results_dir = results_dir
        self.images_dir = images_dir

        # 원본 데이터
        self.train: Optional[pd.DataFrame] = None
        self.test: Optional[pd.DataFrame] = None

        # 전처리된 피처
        self.X_text_train = None      # TF-IDF (train)
        self.X_text_valid = None       # TF-IDF (valid)
        self.X_text_test = None        # TF-IDF (test)
        
        self.X_meta_train = None       # 카테고리+수치 (train)
        self.X_meta_valid = None       # 카테고리+수치 (valid)
        self.X_meta_test = None        # 카테고리+수치 (test)
        
        self.y_train = None            # log1p(price)
        self.y_valid = None

        # 전체 데이터 (최종 학습용)
        self.X_text_full = None
        self.X_meta_full = None
        self.y_full = None

        # 모델
        self.ridge_model: Optional[Ridge] = None
        self.lgbm_model: Optional[LGBMRegressor] = None
        
        # 앙상블 가중치
        self.ensemble_weights = {'ridge': 0.5, 'lgbm': 0.5}  # 초기값
        
        # 인코더/벡터라이저
        self.tfidf = None
        self.label_encoders = {}

        # 메트릭
        self.metrics = {}
    # __init__ end ======================================


    # load_data start ###########################
    def load_data(
        self, 
        train_path: str = '../data/train.tsv',
        test_path: str = '../data/test.tsv',
        sep: str = "\t"
    ):
        """
        데이터 로딩 및 기본 정제
        
        Parameters:
        -----------
        train_path, test_path : str
            데이터 경로
        sep : str
            구분자
        """
        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # 필터링
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])

        # 결측 처리
        for df in [self.train, self.test]:
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["category_name"] = df["category_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")

        print("✅ Data Loaded.")
        print(f"   train: {self.train.shape}, test: {self.test.shape}")
    # load_data end ======================================


    # _split_category start ###########################
    def _split_category(self, df: pd.DataFrame):
        """카테고리 3분할"""
        cats = df["category_name"].str.split("/", n=2, expand=True)
        df["cat1"] = cats[0].fillna("NoCat1")
        df["cat2"] = cats[1].fillna("NoCat2") if cats.shape[1] > 1 else "NoCat2"
        df["cat3"] = cats[2].fillna("NoCat3") if cats.shape[1] > 2 else "NoCat3"
    # _split_category end ======================================


    # preprocess start ###########################
    def preprocess(
        self,
        use_cache: bool = True,
        save_cache: bool = True,
        tfidf_max_features: int = 50000,
        debug: bool = True
    ):
        """
        2-Stage 전략에 맞는 전처리
        
        Parameters:
        -----------
        use_cache : bool
            캐시 사용 여부
        save_cache : bool
            캐시 저장 여부
        tfidf_max_features : int
            TF-IDF 최대 피처 수 (1위 전략: 50000)
        debug : bool
            로그 출력
        
        처리 내용:
        ----------
        1. **텍스트 파이프라인 (Ridge용)**
           - name + description → TF-IDF
           
        2. **메타데이터 파이프라인 (LightGBM용)**
           - category 3분할 → Label Encoding
           - brand_name → Label Encoding
           - item_condition_id, shipping → 그대로 사용
        """
        if self.train is None or self.test is None:
            raise RuntimeError("load_data()를 먼저 호출하세요.")

        cache_dir = os.path.join(self.results_dir, "cache")
        os.makedirs(cache_dir, exist_ok=True)
        cache_path = os.path.join(cache_dir, "mercari_leader_preprocessed_tfidf.pkl")

        # 캐시 로드
        if use_cache and os.path.exists(cache_path):
            if debug:
                print(f"📦 캐시 로드: {cache_path}")
            with open(cache_path, "rb") as f:
                cached = pickle.load(f)
                (
                    self.X_text_train, self.X_text_valid, self.X_text_full, self.X_text_test,
                    self.X_meta_train, self.X_meta_valid, self.X_meta_full, self.X_meta_test,
                    self.y_train, self.y_valid, self.y_full,
                    self.tfidf, self.label_encoders
                ) = cached
            if debug:
                print("✅ 캐시 로드 완료")
            return

        # 카테고리 분할
        self._split_category(self.train)
        self._split_category(self.test)

        # ===== 1. 텍스트 파이프라인 (Ridge용) =====
        if debug:
            print("\n📝 [Stage 1-A] 텍스트 전처리 (Ridge용)")
        
        self.train["text_all"] = (
            self.train["name"].astype(str) + " " +
            self.train["item_description"].astype(str)
        )
        self.test["text_all"] = (
            self.test["name"].astype(str) + " " +
            self.test["item_description"].astype(str)
        )

        self.tfidf = TfidfVectorizer(
            max_features=tfidf_max_features,
            stop_words="english",
            ngram_range=(1, 2)  # 1위 전략: bigram 사용
        )

        X_text_full = self.tfidf.fit_transform(self.train["text_all"])
        X_text_test = self.tfidf.transform(self.test["text_all"])

        if debug:
            print(f"   TF-IDF shape: {X_text_full.shape}")

        # ===== 2. 메타데이터 파이프라인 (LightGBM용) =====
        if debug:
            print("\n🏷️  [Stage 1-B] 메타데이터 전처리 (LightGBM용)")

        # Label Encoding
        cat_cols = ["brand_name", "cat1", "cat2", "cat3"]
        
        for col in cat_cols:
            le = LabelEncoder()
            # train + test 합쳐서 fit
            all_values = pd.concat([self.train[col], self.test[col]]).unique()
            le.fit(all_values)
            
            self.train[f"{col}_encoded"] = le.transform(self.train[col])
            self.test[f"{col}_encoded"] = le.transform(self.test[col])
            
            self.label_encoders[col] = le

        # 메타 피처 구성
        meta_cols = [
            "item_condition_id",
            "shipping",
            "brand_name_encoded",
            "cat1_encoded",
            "cat2_encoded",
            "cat3_encoded"
        ]

        X_meta_full = self.train[meta_cols].astype('int32').values
        X_meta_test = self.test[meta_cols].astype('int32').values

        if debug:
            print(f"   Meta features shape: {X_meta_full.shape}")

        # ===== 3. 타겟 변환 =====
        y_full = np.log1p(self.train["price"].values)

        # ===== 4. Train/Valid Split =====
        indices = np.arange(len(X_text_full))
        train_idx, valid_idx = train_test_split(
            indices,
            test_size=0.2,
            random_state=self.random_state
        )

        # 텍스트
        self.X_text_train = X_text_full[train_idx]
        self.X_text_valid = X_text_full[valid_idx]
        self.X_text_full = X_text_full
        self.X_text_test = X_text_test

        # 메타
        self.X_meta_train = X_meta_full[train_idx]
        self.X_meta_valid = X_meta_full[valid_idx]
        self.X_meta_full = X_meta_full
        self.X_meta_test = X_meta_test

        # 타겟
        self.y_train = y_full[train_idx]
        self.y_valid = y_full[valid_idx]
        self.y_full = y_full

        # 캐시 저장
        if save_cache:
            with open(cache_path, "wb") as f:
                pickle.dump((
                    self.X_text_train, self.X_text_valid, self.X_text_full, self.X_text_test,
                    self.X_meta_train, self.X_meta_valid, self.X_meta_full, self.X_meta_test,
                    self.y_train, self.y_valid, self.y_full,
                    self.tfidf, self.label_encoders
                ), f)
            if debug:
                print(f"\n💾 캐시 저장: {cache_path}")

        if debug:
            print("\n✅ 전처리 완료")
            print(f"   Train: {len(train_idx):,}, Valid: {len(valid_idx):,}")
    # preprocess end ======================================


    # train_ridge start ###########################
    def train_ridge(self, alpha: float = 0.5, verbose: bool = True):
        """
        Stage 1-A: Ridge Regressor 학습 (텍스트 전용)
        
        Parameters:
        -----------
        alpha : float
            Ridge 정규화 파라미터 (1위 전략: 0.5)
        verbose : bool
            로그 출력
        
        Returns:
        --------
        dict : 메트릭
        """
        if self.X_text_train is None:
            raise RuntimeError("preprocess()를 먼저 호출하세요.")

        if verbose:
            print("\n" + "="*80)
            print("  [Stage 1-A] Ridge Regressor 학습 (텍스트)")
            print("="*80)

        self.ridge_model = Ridge(alpha=alpha, random_state=self.random_state)
        self.ridge_model.fit(self.X_text_train, self.y_train)

        # 예측
        y_pred_log = self.ridge_model.predict(self.X_text_valid)
        
        # 메트릭 계산 (원래 스케일)
        y_true = np.expm1(self.y_valid)
        y_pred = np.maximum(np.expm1(y_pred_log), 0)

        metrics = self._calculate_metrics(y_true, y_pred)

        if verbose:
            print(f"✓ Ridge 학습 완료")
            print(f"  RMSLE: {metrics['rmsle']:.6f}")
            print(f"  RMSE:  {metrics['rmse']:.2f}")

        return metrics
    # train_ridge end ======================================


    # train_lgbm start ###########################
    def train_lgbm(
        self,
        params: Optional[dict] = None,
        verbose: bool = True
    ):
        """
        Stage 1-B: LightGBM 학습 (메타데이터 전용)
        
        Parameters:
        -----------
        params : dict, optional
            LightGBM 하이퍼파라미터 (None이면 기본값)
        verbose : bool
            로그 출력
        
        Returns:
        --------
        dict : 메트릭
        """
        if self.X_meta_train is None:
            raise RuntimeError("preprocess()를 먼저 호출하세요.")

        if verbose:
            print("\n" + "="*80)
            print("  [Stage 1-B] LightGBM 학습 (메타데이터)")
            print("="*80)

        # 기본 파라미터 (1위 전략 기반)
        if params is None:
            params = {
                'n_estimators': 3000,
                'learning_rate': 0.75,
                'num_leaves': 31,
                'max_depth': -1,
                'min_child_samples': 20,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'random_state': self.random_state,
                'n_jobs': -1,
                'verbose': -1
            }

        self.lgbm_model = LGBMRegressor(**params)
        self.lgbm_model.fit(
            self.X_meta_train,
            self.y_train,
            eval_set=[(self.X_meta_valid, self.y_valid)],
            callbacks=[
                LGBMRegressor().early_stopping(50, verbose=False)
            ] if verbose else None
        )

        # 예측
        y_pred_log = self.lgbm_model.predict(self.X_meta_valid)
        
        # 메트릭
        y_true = np.expm1(self.y_valid)
        y_pred = np.maximum(np.expm1(y_pred_log), 0)

        metrics = self._calculate_metrics(y_true, y_pred)

        if verbose:
            print(f"✓ LightGBM 학습 완료")
            print(f"  RMSLE: {metrics['rmsle']:.6f}")
            print(f"  RMSE:  {metrics['rmse']:.2f}")

        return metrics
    # train_lgbm end ======================================


    # optimize_ensemble start ###########################
    def optimize_ensemble(self, verbose: bool = True):
        """
        Stage 2: 앙상블 가중치 최적화
        
        Ridge와 LightGBM 예측을 조합하는 최적 가중치 탐색
        가중치 범위: 0.0 ~ 1.0 (0.05 간격)
        
        Returns:
        --------
        dict : {'ridge': w1, 'lgbm': w2, 'rmsle': score}
        """
        if self.ridge_model is None or self.lgbm_model is None:
            raise RuntimeError("train_ridge()와 train_lgbm()를 먼저 호출하세요.")

        if verbose:
            print("\n" + "="*80)
            print("  [Stage 2] 앙상블 가중치 최적화")
            print("="*80)

        # 각 모델의 예측값 (log scale)
        ridge_pred_log = self.ridge_model.predict(self.X_text_valid)
        lgbm_pred_log = self.lgbm_model.predict(self.X_meta_valid)

        best_rmsle = float('inf')
        best_weights = None

        # Grid search (0.0 ~ 1.0, 0.05 간격)
        for ridge_weight in np.arange(0.0, 1.05, 0.05):
            lgbm_weight = 1.0 - ridge_weight
            
            # 가중 평균 (log scale에서)
            ensemble_pred_log = (
                ridge_weight * ridge_pred_log +
                lgbm_weight * lgbm_pred_log
            )
            
            # 원래 스케일로 변환
            y_true = np.expm1(self.y_valid)
            y_pred = np.maximum(np.expm1(ensemble_pred_log), 0)
            
            # RMSLE 계산
            rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))
            
            if rmsle < best_rmsle:
                best_rmsle = rmsle
                best_weights = {'ridge': ridge_weight, 'lgbm': lgbm_weight}

        self.ensemble_weights = best_weights

        if verbose:
            print(f"✓ 최적 가중치 발견")
            print(f"  Ridge:  {best_weights['ridge']:.2f}")
            print(f"  LightGBM: {best_weights['lgbm']:.2f}")
            print(f"  RMSLE:  {best_rmsle:.6f}")

        return {**best_weights, 'rmsle': best_rmsle}
    # optimize_ensemble end ======================================


    # evaluate start ###########################
    def evaluate(self, verbose: bool = True):
        """
        최종 앙상블 평가
        
        Returns:
        --------
        dict : 전체 메트릭
        """
        if self.ridge_model is None or self.lgbm_model is None:
            raise RuntimeError("모델을 먼저 학습하세요.")

        ridge_pred_log = self.ridge_model.predict(self.X_text_valid)
        lgbm_pred_log = self.lgbm_model.predict(self.X_meta_valid)

        ensemble_pred_log = (
            self.ensemble_weights['ridge'] * ridge_pred_log +
            self.ensemble_weights['lgbm'] * lgbm_pred_log
        )

        y_true = np.expm1(self.y_valid)
        y_pred = np.maximum(np.expm1(ensemble_pred_log), 0)

        self.metrics = self._calculate_metrics(y_true, y_pred)

        if verbose:
            print("\n" + "="*80)
            print("  최종 평가 (Ensemble)")
            print("="*80)
            for k, v in self.metrics.items():
                print(f"  {k.upper()}: {v:.6f}" if k == 'rmsle' else f"  {k.upper()}: {v:.4f}")

        return self.metrics
    # evaluate end ======================================


    # _calculate_metrics start ###########################
    def _calculate_metrics(self, y_true, y_pred):
        """메트릭 계산 헬퍼"""
        return {
            'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
            'mae': float(mean_absolute_error(y_true, y_pred)),
            'r2': float(r2_score(y_true, y_pred)),
            'rmsle': float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)))
        }
    # _calculate_metrics end ======================================


    # predict_test start ###########################
    def predict_test(self, save_submission: bool = True):
        """
        테스트 데이터 예측 및 제출 파일 생성
        
        Parameters:
        -----------
        save_submission : bool
            submission CSV 저장 여부
        
        Returns:
        --------
        np.ndarray : 예측 가격
        """
        if self.ridge_model is None or self.lgbm_model is None:
            raise RuntimeError("모델을 먼저 학습하세요.")

        # 각 모델 예측
        ridge_pred_log = self.ridge_model.predict(self.X_text_test)
        lgbm_pred_log = self.lgbm_model.predict(self.X_meta_test)

        # 앙상블
        ensemble_pred_log = (
            self.ensemble_weights['ridge'] * ridge_pred_log +
            self.ensemble_weights['lgbm'] * lgbm_pred_log
        )

        y_pred = np.maximum(np.expm1(ensemble_pred_log), 0)

        # Submission 저장
        if save_submission:
            os.makedirs(self.results_dir, exist_ok=True)
            
            if "test_id" in self.test.columns:
                ids = self.test["test_id"].values
            elif "id" in self.test.columns:
                ids = self.test["id"].values
            else:
                ids = np.arange(len(self.test))

            sub_df = pd.DataFrame({"test_id": ids, "price": y_pred})
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"submission_leader_{timestamp}.csv"
            save_path = os.path.join(self.results_dir, filename)
            
            sub_df.to_csv(save_path, index=False)
            print(f"📄 Submission 저장: {save_path}")

        return y_pred
    # predict_test end ======================================


    # save_models start ###########################
    def save_models(self):
        """모델 저장"""
        os.makedirs(self.models_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # Ridge
        ridge_path = os.path.join(self.models_dir, f"ridge_leader_{timestamp}.pkl")
        with open(ridge_path, 'wb') as f:
            pickle.dump(self.ridge_model, f)

        # LGBM
        lgbm_path = os.path.join(self.models_dir, f"lgbm_leader_{timestamp}.pkl")
        with open(lgbm_path, 'wb') as f:
            pickle.dump(self.lgbm_model, f)

        # 앙상블 정보
        ensemble_info = {
            'weights': self.ensemble_weights,
            'metrics': self.metrics,
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        ensemble_path = os.path.join(self.results_dir, f"ensemble_info_{timestamp}.json")
        with open(ensemble_path, 'w') as f:
            json.dump(ensemble_info, f, indent=4)

        print(f"💾 모델 저장 완료:")
        print(f"   Ridge: {ridge_path}")
        print(f"   LGBM:  {lgbm_path}")
        print(f"   Info:  {ensemble_path}")
    # save_models end ======================================


print("✅ MercariLeaderAnalyzer 로드 완료!")
print("""
사용법:
  analyzer = MercariLeaderAnalyzer()
  analyzer.load_data()
  analyzer.preprocess(tfidf_max_features=50000)
  analyzer.train_ridge(alpha=0.5)
  analyzer.train_lgbm()
  analyzer.optimize_ensemble()
  analyzer.evaluate()
  analyzer.predict_test()
""")

In [ ]:
analyzer = MercariLeaderAnalyzer()
analyzer.load_data()
analyzer.preprocess(tfidf_max_features=50000)
analyzer.train_ridge(alpha=0.5)
analyzer.train_lgbm()
analyzer.optimize_ensemble()
analyzer.evaluate()
analyzer.predict_test()